In [ ]:
import numpy as np
#import matplotlib
#%matplotlib notebook
import matplotlib.pyplot as plt
import pylcp
from pylcp.fields import gaussianBeam
import cProfile, pstats, io

In [ ]:
# 50 gauss/cm for Sr 
# mub on database
# 0.43 cm for x0 30/1.4/50

# x is length unit
# k is wavenumber which relates to wavelength
x0 = (30/1.4/50) # cm
k = 2*np.pi/461E-7 # cm^{-1}
kbar = k*x0

# D source for position
d = np.array([0., -8.839/10, -8.839/10])/x0

# Gamma is decay rate
# t0 is normalized time units of decary
# wb is width factor
gamma = 2*np.pi*30e6
t0 = 1e-5 # s
gammabar = gamma*t0
wb = 4.7 # mm

In [ ]:
nr=3

class aPHIMOT(pylcp.laserBeams):
     def __init__(self, *args, **kwargs):
        super().__init__()
    
        beam_type = kwargs.pop('beam_type', pylcp.laserBeam)
        pol = kwargs.pop('pol', +1)
        kmag = kwargs.pop('k', 1.)
        
        self.add_laser(beam_type(kmag*(np.array([0., 0.,  17.678])/(np.linalg.norm(np.array([0., 0.,  17.678])))), +pol, *args, **kwargs))
        self.add_laser(beam_type(kmag*(np.array([0., 0.,  -17.678])/(np.linalg.norm(np.array([0., 0.,  -17.678])))), +pol, *args, **kwargs))
        self.add_laser(beam_type(kmag*(np.array([10.825, -13.258, 4.419])/(np.linalg.norm(np.array([10.825, -13.258, 4.419])))), -pol, *args, **kwargs))
        self.add_laser(beam_type(kmag*(np.array([10.825, 13.258, -4.419])/(np.linalg.norm(np.array([10.825, 13.258, -4.419])))), -pol, *args, **kwargs))
        self.add_laser(beam_type(kmag*(np.array([-10.825, -13.258, 4.419])/(np.linalg.norm(np.array([-10.825, -13.258, 4.419])))), -pol, *args, **kwargs))
        self.add_laser(beam_type(kmag*(np.array([-10.825, 13.258, -4.419])/(np.linalg.norm(np.array([-10.825, 13.258, -4.419])))), -pol, *args, **kwargs))
        
class rotDivGauss(pylcp.laserBeams):
     def __init__(self, *args, **kwargs):
        super().__init__()
    
        beam_type = kwargs.pop('beam_type', pylcp.laserBeam)
        pol = kwargs.pop('pol', +1)
        kmag = kwargs.pop('k', 1.)
        beta = kwargs.pop('beta', 1.0)
        delta = kwargs.pop('delta', -1.5*gammabar)
        kmag = 2*np.pi/461e-6
        
        self.add_laser(pylcp.gaussianBeam(kvec=kmag*np.array([0., 0., -1.]), \
                       pol=np.array([1., 1.j, 0])/np.sqrt(2), beta=1.0*beta, delta=delta, wb=4.65, r0=np.array([0., 0., 0.]), **kwargs))
        self.add_laser(pylcp.gaussianBeam(kvec=kmag*np.array([0.612, 0.75, -0.25]), \
                       pol=np.array([1., -1.j, 0])/np.sqrt(2), beta=1.0*beta, delta=delta, wb=4.65, r0=np.array([0., 0., 0.]), **kwargs))
        self.add_laser(pylcp.gaussianBeam(kvec=kmag*np.array([-0.612, 0.75, -0.25]), \
                       pol=np.array([1., -1.j, 0])/np.sqrt(2), beta=1.0*beta, delta=delta, wb=4.65, r0=np.array([0., 0., 0.]), **kwargs))
        self.add_laser(pylcp.gaussianBeam(kvec=kmag*np.array([0., 0., 1.]), \
                      pol=np.array([1., 1.j, 0])/np.sqrt(2), beta=6.261e7*beta, delta=delta, wb=5.744e-4, r0=np.array([0., 0., -17.788]), **kwargs))
        self.add_laser(pylcp.gaussianBeam(kvec=kmag*np.array([-0.612, -0.75, 0.25]), \
                       pol=np.array([1., -1.j, 0])/np.sqrt(2), beta=6.261e7*beta, delta=delta, wb=5.744e-4, r0=np.array([10.893, 13.341, -4.447]),**kwargs))
        self.add_laser(pylcp.gaussianBeam(kvec=kmag*np.array([0.612, -0.75, 0.25]), \
                       pol=np.array([1., -1.j, 0])/np.sqrt(2), beta=6.261e7*beta, delta=delta, wb=5.744e-4, r0=np.array([-10.893, 13.341, -4.447]), **kwargs))
    

# Set up the laser beams with their appropriate characteristics
det = -1.5*gammabar
beta = 1.0

#beam_to_sim = pylcp.infinitePlaneWaveBeam
beam_to_sim = pylcp.gaussianBeam
#beam_to_sim = pylcp.clippedGaussianBeam

#MOT_to_sim = conventional3DMOTBeams
#MOT_to_sim = aPHIMOT
MOT_to_sim = rotDivGauss

#MOT_to_sim_kwargs = {'rotation_angles':[np.pi/4, 0., 0.]} # Extra arguments for conventional3DMOTBeams
MOT_to_sim_kwargs = {} # Extra arguments for aPHIMOT/rotDivGauss

laser_kwargs = {} # Use for rotDivGauss
#laser_kwargs = {'wb':wb} # Extra arguments for GaussianBeam
#laser_kwargs = {'wb':1000*wb, 'rs':wb} # Extra arguments for clippedGaussianBeam

testBeams = MOT_to_sim(beta=beta, delta=det, beam_type=beam_to_sim, k=kbar, **laser_kwargs, **MOT_to_sim_kwargs) # kbar

############################################################################################

#Trigger numba to compile the beta code:
testBeams.beam_vector[0].beta()
testBeams.beam_vector[1].beta()

x_beta = 15
X, Y = np.meshgrid(np.linspace(-x_beta, x_beta, 101),
                   np.linspace(-x_beta, x_beta, 101))
z_tests = [-4, 0, +4]

plt.figure("Laser Beams", figsize=(4, 1.5*nr))
plt.clf()
pr = cProfile.Profile()

for jj, laserBeam in enumerate(testBeams.beam_vector):
    for ii, z_test in enumerate(z_tests):
        Z = z_test*np.ones(X.shape)
        it = np.nditer([X, Y, Z, None])
        Rt=np.array([X, Y, Z])

        """pr.enable()
        for (x, y, z, beta) in it:
            beta[...] = laserBeam.beta(np.array([x, y, z]), 0.)
        pr.disable()
        
        BETA = it.operands[3]"""
        
        pr.enable()
        BETA = laserBeam.beta(Rt)
        pr.disable()
        
        plt.subplot(len(testBeams.beam_vector), len(z_tests), jj*len(z_tests)+ii+1)
        plt.imshow(BETA, origin='lower',
                   extent=(-x_beta, x_beta,
                           -x_beta, x_beta))
        plt.clim((0, 1))
        plt.set_cmap('jet')
        # Make a cross-hair:
        plt.plot([0, 0], [-x_beta, x_beta],
                 'w-', linewidth=0.25)
        plt.plot([-x_beta, x_beta], [0, 0],
                 'w-', linewidth=0.25)

In [ ]:
zchip = 1.0
Z = (zchip-8)*np.ones(X.shape)

Rt=np.array([X, Y, Z])
plt.imshow(np.sum(testBeams.beta(Rt)[1::], axis=0))

In [ ]:
# s = io.StringIO()
# sortby = 'cumtime'
# ps = pstats.Stats(pr, stream=s).sort_stats(sortby)
# ps.print_stats()
# print(s.getvalue())

In [ ]:
91808+30603

In [ ]:
nr=3
# testBeams = gratings.infiniteGratingMOTBeams(delta=-1., s=1., nr=nr, thd=np.pi/4,
#                                              pol=np.array([-1/np.sqrt(2), 1j/np.sqrt(2), 0]),
#                                              reflected_pol=np.array([np.pi, 0]),
#                                              reflected_pol_basis='poincare', eta=None, grating_angle=0)
x_beta = 2
X, Y = np.meshgrid(np.linspace(-x_beta, x_beta, 101),
                   np.linspace(-x_beta, x_beta, 101))
z_tests = [0]

plt.figure("Infinite Beams", figsize=(4, 1.5*nr))
plt.clf()
for jj, laserBeam in enumerate(testBeams.beam_vector):
    for ii, z_test in enumerate(z_tests):
        plt.subplot(len(testBeams.beam_vector), len(z_tests), jj*len(z_tests)+ii+1)
        Rt=np.array([X, Y, z_test*np.ones(X.shape)])
        tt=np.zeros(X.shape)
        print(laserBeam.kvec())
        plt.imshow(laserBeam.beta(R=Rt,t=tt),
                   origin='lower',
                   extent=(-x_beta, x_beta,
                           -x_beta, x_beta))
        plt.clim((0, 1))
        plt.set_cmap('jet')
        # Make a cross-hair:
        plt.plot([0, 0], [-x_beta, x_beta],
                 'w-', linewidth=0.25)
        plt.plot([-x_beta, x_beta], [0, 0],
                 'w-', linewidth=0.25)

In [ ]:
# a = [1,2,3,4]
# print(a)
# print(a[1::])
# print(a[1::2])
# print(a[0::])
# print(a[:-2])
# print(a[:-1:2])